## Actividad 3_19: Arroz
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para solucionar el problema de clasificar tipos de arroz.
    <ol>
        <li>Descarga el archivo "Rice_Image_Dataset.zip" de <a href="https://www.muratkoklu.com/datasets/vtdhnd09.php">https://www.muratkoklu.com/datasets/vtdhnd09.php</a>. Descomprime el archivo y guarda la carpeta en un lugar adecuado.</li>
        <li>Importa los datos usando las misma técnica que en la actividad de los pistachos (En este caso, 250x250 en escala de grises).</li>
        <li>Guarda las etiquetas de los datos. Debes tener un conjunto photos y otro labels que estén ordenados igual. labels debe tener números entre 0.0 y 4.0, ya que hay 5 clases de arroz.</li>
        <li>Utiliza PCA para reducir el dataset.</li>
        <li>Soluciona el ejercicio con una red neuronal. Puedes utilizar todas las técnicas que hemos aprendido.</li>
        <li>Soluciona el ejercicio usando una red convolucional.</li>
    </ol>
</div>

In [2]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./Rice_Image_Dataset')
#Clase Kirmizi será la clase 0.0
#Clase Siirt será la clase 1.0


photos =  []
labels = []

I0000 00:00:1775499131.707057   29508 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775499136.045691   29508 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775499146.240951   29508 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
#2. IMPORTAMOS LOS DATOS:
for idx,folder in enumerate(folders):
    for file in listdir('./Rice_Image_Dataset/'+folder):
        #Cargamos la imagen.
        #load_img sirve para cargar las imágenes en memoria. Tiene distintos parámetros para modificar como se cargan las imágenes.
        photo = load_img('./Rice_Image_Dataset/'+folder+'/' + file, color_mode='grayscale') 
        #Convertimos la imagen a un array.
        photo = img_to_array(photo)
        #Los guardamos en las listas.
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

0


KeyboardInterrupt: 

In [ ]:
photos = asarray(photos)
labels = asarray(labels)

In [ ]:
photos.shape

(75000, 250, 250, 1)

In [ ]:
labels.shape

(75000,)

In [ ]:
photos = photos / 255.0
X = photos
X = X.reshape(75000, -1)
y = labels
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
X, y = photos, labels
print(X.shape)
print(y.shape)

(75000, 250, 250, 1)
(75000,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

: 

In [ ]:
plt.figure(figsize=(28,28))
for i in range(16):
    some_digit = X_train.iloc[[i]].values
    some_digit_image = some_digit.reshape(28, 28)
    plt.subplot(4,4,i+1)
    plt.imshow(some_digit_image, cmap="binary")
    plt.axis("off")
plt.show()

NameError: name 'plt' is not defined

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
pca.fit(X_train)

In [ ]:
print(pca.explained_variance_ratio_)

In [ ]:
cumsum = np.cumsum(pca.explained_variance_ratio_)

In [ ]:
print(cumsum)

In [ ]:
plt.plot(range(len(cumsum)),cumsum,color='blue')
plt.title("Varianza acumulada vs número de dimensiones")
plt.xlabel('Dimensiones')
plt.ylabel('Varianza')
plt.show()

In [ ]:
d = np.argmax(cumsum >= 0.99999) + 1
print(d)

In [ ]:
#pca = PCA(n_components=d)
#NOTA: ESTO LO HEMOS HECHO PARA ENTENDER PCA. A PARTIR DE AHORA LO HAREMOS ASÍ:
pca = PCA(n_components=0.95)
#Si en n_components metes un float menor que uno, es directamente la varianza que quieres. Todo lo de arriba no
# habría que hacerlo.
pca.fit(X_train)
X_reduced = pca.transform(X_train)
X_recovered = pca.inverse_transform(X_reduced)
X_reduced.shape

#Esto lo hemos hecho para ver paso a paso lo que hace el algoritmo, se podría hacer más rápido metiendo 
# en n_components un valor float entre 0.0 y 1.0. 

In [ ]:
X_test_reduced = pca.transform(X_test)

In [ ]:
plt.figure(figsize=(28,28))
for i in range(16):
    some_digit = X_recovered[i]
    some_digit_image = some_digit.reshape(28, 28)
    plt.subplot(4,4,i+1)
    plt.imshow(some_digit_image, cmap="binary")
    plt.axis("off")
plt.show()

In [ ]:
#El tipo de activación se saca de la tabla de teoría.
model = keras.models.Sequential([
keras.layers.Dense(512, activation="relu", input_shape=X_train.shape[1:]),
keras.layers.Dense(256, activation="relu"),
keras.layers.Dense(128, activation="relu"),
keras.layers.Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer= keras.optimizers.SGD(learning_rate=0.001),metrics=['accuracy'])